# 🤝 vdclient sdk 使用教學(操作 SFTP 與 Mongo)

* 取得資料源連線資訊

| 類別/模組 | 功能 | 備註 |
|-----------|------|------|
| `get_datasource_config(datasource)` | 取得完整連線設定 | 使用 DeepFlow SDK 取得連線設定 |


* 取得資料源連線 connection 
| 類別/模組 | 功能 | 備註 |
|-----------|------|------|
| `rdb_connection(datasource)` | 回傳 SQLAlchemy connection | 支援 PostgreSQL, SQL Server, Oracle |
| ✅`sftp_client(datasource)` | 回傳 Paramiko SFTPClient | 可用於下載、上傳、列出遠端檔案 |
| ✅`mongo_client(datasource)` | 回傳 pymongo.MongoClient | 支援 Mongo 資料庫的操作 |


## 🔐 匯入必要模組與建立連線

In [1]:
from vdclient_magic.core.connectors import sftp_client, sftp_client_from_config
import os
from datetime import datetime

## 📁 SFTP 操作範例（建立暫存目錄 + 上傳下載測試檔案）

In [2]:
# 建立連線
sftp = sftp_client("nas")  # 在 DeepFlow 資料源連線管理上設定的「資料源名稱」

In [3]:
type(sftp)

paramiko.sftp_client.SFTPClient

In [4]:
# 顯示根目錄檔案
files = sftp.listdir(".")
print("遠端檔案清單：", files)

遠端檔案清單： ['.conda', 'miniconda3', '.bashrc', '.profile', '.bash_logout', '__pycache__', '.python_history', '.ssh', 'app.py', '.wget-hsts', '.bash_history', '.cache', '.viminfo', 'pypi', 'Miniconda3-py311_25.1.1-2-Linux-x86_64.sh']


In [5]:
# 建立一個暫存資料夾
temp_dir = "temp_demo"
sftp.mkdir(temp_dir)
print(f"已建立遠端目錄：{temp_dir}")

已建立遠端目錄：temp_demo


In [6]:
# 建立一個本地測試檔案
local_file = "/tmp/demo_test.txt"
with open(local_file, "w") as f:
    f.write(f"This is a demo file created at {datetime.now()}\n")
print(f"已建立本地檔案：{local_file}")

已建立本地檔案：/tmp/demo_test.txt


In [7]:
# 上傳到 temp_folder 中
remote_path = 'temp_demo/demo_test.txt'
sftp.put(local_file, remote_path)
print(f"已上傳至：{remote_path}")


已上傳至：temp_demo/demo_test.txt


In [8]:
# 將遠端檔案重新命名
renamed_path = "temp_demo/renamed_test.txt"
sftp.rename(remote_path, renamed_path)
print(f"已重新命名檔案為：{renamed_path}")


已重新命名檔案為：temp_demo/renamed_test.txt


In [9]:
# 下載回來作驗證
downloaded_file = "/tmp/downloaded_test.txt"
sftp.get(renamed_path, downloaded_file)
print(f"已下載檔案為：{downloaded_file}")

# 顯示檔案內容
with open(downloaded_file) as f:
    print(f.read())


已下載檔案為：/tmp/downloaded_test.txt
This is a demo file created at 2025-08-18 18:06:40.853530



In [10]:
# 清理：刪除遠端測試檔案與目錄
sftp.remove(renamed_path)
sftp.rmdir(temp_dir)
print("已刪除遠端測試檔與目錄")

# 清理本地檔案
os.remove(local_file)
os.remove(downloaded_file)

# 關閉連線
sftp.close()


已刪除遠端測試檔與目錄


## 🍃 Mongo 操作範例（使用測試 collection）

In [11]:
from vdclient_magic.core.connectors import mongo_client
import pandas as pd

In [12]:
# 建立連線與取得資料庫 / 集合
mongo = mongo_client("mongo")      # 在 DeepFlow 資料源連線管理上設定的「資料源名稱」

In [13]:
type(mongo)

pymongo.synchronous.mongo_client.MongoClient

In [14]:
db = mongo["mydb"]                 # 取得資料庫物件（Database）
collections = db.list_collection_names()  # ✅ 取得該資料庫中所有 collection 名稱
print(collections)

['trino_schema', 'provider_data', 'custom_report', 'inventory']


In [15]:
collection = db["temp_orders"]

In [16]:
# 建立連線與測試 collection
collection = db["temp_orders"]

# 插入一些測試資料
collection.insert_many([
    {"order_id": 1001, "amount": 500, "status": "pending"},
    {"order_id": 1002, "amount": 1500, "status": "completed"},
])
print("已插入測試資料")


已插入測試資料


In [17]:
# 查詢測試資料並轉為 DataFrame
docs = list(collection.find({}))
df = pd.DataFrame(docs)
df


,_id,order_id,amount,status
0,68a2fb31ce5e4ccd3c04e3f6,1001,500,pending
1,68a2fb31ce5e4ccd3c04e3f7,1002,1500,completed


In [18]:
# 更新與刪除資料測試
collection.update_one({"order_id": 1001}, {"$set": {"status": "shipped"}})
collection.delete_one({"order_id": 1002})
print("已完成更新與刪除操作")


已完成更新與刪除操作


In [19]:
# 最後清空整個測試 collection
collection.drop()
print("已刪除測試 collection")
mongo.close()


已刪除測試 collection
